In [1]:
from skidl.logger import stop_log_file_output
stop_log_file_output(True)

In [2]:
from python.spice_tools import search_spice_model, save_part_model

entry = search_spice_model(name="NTCG163JF103FTDS", library="Sensors")
if entry:
    print(entry["model_content"])
else:
    print("Not in model db")
 

* NTCG163JF103FTDS - TDK NTCG series NTC thermistor, 1608 (EIA 0603), automotive 125C grade
* 10 kOhm at 25C, B-constant (25/85C) = 3435 K, (25/50C) = 3380 K, (25/75C) = 3422 K, (25/100C) = 3453 K
* Operating temperature range (part): -40C to +125C (catalog, NTCG1608 125C grade)
* Maximum rated power at 25C: 100 mW; thermal dissipation constant at 25C: 1 mW/C (NTCG1608, 125C grade table)
* Permissible operating current at 25C: 0.31 mA (table for NTCG163JF103FTDS)
*
* Pins (must match provided pinout):
*   1 : terminal 1
*   2 : terminal 2
*
* Modeling approach
* -----------------
* - Resistance is computed from the datasheet equation R(T) = R25 * exp( B * (1/T - 1/Tref) ).
* - A single effective B (default BETA = 3435 K, the 25/85C value) is used.
* - Self-heating and thermal dynamics are not modeled; resistance follows the parameter TAMB.
*
* Usage notes
* -----------
* - Set TAMB (degC) via a .param or .step command to represent the thermistor temperature.
* - The model does not enfo

In [3]:
   # Save part model

with open("test_cases/test_charger_3A/spice/battery_temperature_sense/NTCG163JF103FTDS.spice.lib", "r") as f:
    model_content = f.read()
save_part_model(
name="NTCG163JF103FTDS",
library="Sensors",
model_content=model_content,
vendor_provided=False,
)

In [4]:
# from pathlib import Path
# from python.spice_tools import convert_skidl_module

# name = convert_skidl_module(
#     input_path=Path("test_cases/test_charger_3A/skidl/modules/ip2312_charger.py"),
#     subckt_name="IP2312_CHARGER",
#     output_path=Path("test_cases/test_charger_3A/spice/ip2312_charger/dut.py"),
# )

# name

In [5]:
import json

test_bench_path = "test_cases/case_3A_charger/testbench/battery_protection_schema_valid.json"

with open(test_bench_path, "r") as f:
    test_bench = json.load(f)
    testcases = test_bench["use_cases"]
case_ids = [case["name"] for case in testcases]

case_ids

['startup_reverse_block_with_charger_ramp',
 'forward_conduction_drop_3A_25C',
 'forward_conduction_drop_3A_60C',
 'reverse_blocking_usb_removed',
 'overcharge_overvoltage_cutoff',
 'overdischarge_cutoff_under_load',
 'discharge_short_circuit_protection',
 'charge_direction_short_circuit_protection',
 'steady_ripple_pass_through']

In [6]:
from python.spice_tools.harness_sanity import harness_sanity_check

for idx in range(len(case_ids)):
    harness_path = f"test_cases/case_3A_charger/spice/battery_protection/testcases/{case_ids[idx]}.py"

    result = harness_sanity_check(harness_path=harness_path)
    print(result)


{'ok': True, 'max_abs_voltage': 3.179966143196661, 'max_abs_current': 15.600001559999889, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 3.9374998046875103, 'max_abs_current': 0.0, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 0.0, 'max_abs_current': 0.0, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 2.924999878139598, 'max_abs_current': 19.50000243772368, 'num_points': 526, 'error': None}
{'ok': True, 'max_abs_voltage': 4.2851271236494615, 'max_abs_current': 0.0, 'num_points': 520, 'error': None}
{'ok': True, 'max_abs_voltage': 1.8333332722222246, 'max_abs_current': 3.0, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 3.8999999512500017, 'max_abs_current': 24.375100492934543, 'num_points': 523, 'error': None}
{'ok': True, 'max_abs_voltage': 4.124999795575323, 'max_abs_current': 1.5000040884935568, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 3.0125132697252983, 'max_abs_current': 0.0, 'nu

In [7]:
from python.spice_tools.testbench_runner import run_use_case

for idx in range(len(case_ids)):
    harness_path = f"test_cases/case_3A_charger/spice/battery_protection/testcases/{case_ids[idx]}.py"

    reports = run_use_case(
        schema_path=test_bench_path,
        harness_path=harness_path,
        use_case_name=case_ids[idx],
        dut_path="test_cases/case_3A_charger/spice/battery_protection/battery_protection_pyspice.py",
        dut_module_name="Battery_Protection_pyspice",
    )

    print(reports)


{'startup_reverse_block_with_charger_ramp': {'total_measurements': 3, 'num_passed': 3, 'all_passed': True, 'measurements': {'reverse_leakage_during_ramp': {'assertion': {'value': -1.1556215201815248e-06, 'op': '<=', 'limit': 0.001}, 'passed': True}, 'pack_overshoot_on_connect': {'assertion': {'value': 1.2458574349949458e-07, 'op': '<=', 'limit': 0.05}, 'passed': True}, 'pack_voltage_in_band': {'assertion': {'value': 1.0, 'op': '>=', 'limit': 0.99}, 'passed': True}}}}
{'forward_conduction_drop_3A_25C': {'total_measurements': 2, 'num_passed': 0, 'all_passed': False, 'measurements': {'vdrop_mean_3A_25C': {'assertion': {'value': 0.24999999987569013, 'op': '<=', 'limit': 0.12}, 'passed': False}, 'module_efficiency_3A_25C': {'assertion': {'value': 93.74999999998296, 'op': '>=', 'limit': 97.0}, 'passed': False}}}}


KeyError: 'VBAT_CHG'

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "python"))
from commands.database_tools.library_schematic import LibraryManager
from python.spice_tools.utils import validate_spice_model

mpn = "NTCG163JF103FTDS"
lib = "Sensors"

result = LibraryManager.get_symbol_pinout({
    "library": lib,
    "symbol": mpn,
})

pins = result.get("pins")
print(pins)

res = validate_spice_model(
    model_path="/root/workspace/KiCAD_MCP/test_cases/test_charger_3A/spice/battery_temperature_sense/NTCG163JF103FTDS.spice.lib",
    expected_subckt_name=mpn,
    expected_pinout=pins,
)

In [ ]:
for r in res:
    print(r+'\n\n')